In [1]:
import os
import torch
import mlflow
import dagshub
from ultralytics import YOLO
from ultralytics import settings

In [2]:
torch.cuda.empty_cache()

In [3]:
MY_USERNAME = "TQanh23" 
REPO_NAME = "ppe_v2"

In [4]:
dagshub.init(repo_owner=MY_USERNAME, repo_name=REPO_NAME, mlflow=True)
settings.update({'mlflow': True})

Accessing as TQanh23

Initialized MLflow to track repo "TQanh23/ppe_v2"

Repository TQanh23/ppe_v2 initialized!

In [6]:
mlflow.set_experiment("PPE__yolov8m_150e_Baseline_Evaluation")

with mlflow.start_run(run_name="YOLOv8m_Baseline_Val") as run:
    
    # 3. Load mô hình (Đảm bảo PyTorch đã nhận CUDA như hướng dẫn trước đó)
    model_path = 'downloads/best.pt'
    model = YOLO(model_path)
    
    print("Đang tiến hành đánh giá và log dữ liệu lên DagsHub...")
    
    # 4. Chạy quá trình Validation
    # Ultralytics sẽ tự động log các thông số (parameters) và độ chính xác (metrics) vào MLflow
    metrics = model.val(
        data='data.yaml',
        device=0,           # Sử dụng GPU RTX 3050
        batch=4,
        imgsz=640,
        split='val',
        project='PPE_Detection',
        name='baseline_evaluation'
    )

2026/07/25 13:44:35 INFO mlflow.tracking.fluent: Experiment with name 'PPE__yolov8m_150e_Baseline_Evaluation' does not exist. Creating a new experiment.


Đang tiến hành đánh giá và log dữ liệu lên DagsHub...
Ultralytics 8.4.104  Python-3.12.7 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
Model summary (fused): 93 layers, 25,844,971 parameters, 0 gradients, 78.7 GFLOPs
WARNING val: Slow image access detected (ping: 0.10.0 ms, read: 3.50.8 MB/s, size: 68.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning D:\ppe_cv\train_data\valid\labels.cache... 229 images, 19 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 229/229  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 58/58 7.0it/s 8.2s0.1s
                   all        229       2019      0.743      0.689      0.716      0.357
                 boots         93        201      0.584      0.567      0.549      0.231
                gloves         78        157      0.639      0.451      0.481      0.18

In [7]:
print("Đang upload file trọng số lên DagsHub...")
mlflow.log_artifact(model_path, artifact_path="models")

# In kết quả ra màn hình local
print("\n--- KẾT QUẢ BASELINE ĐÃ LOG LÊN DAGSHUB ---")
print(f"Run ID:       {run.info.run_id}")
print(f"mAP@0.5:      {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")

Đang upload file trọng số lên DagsHub...

--- KẾT QUẢ BASELINE ĐÃ LOG LÊN DAGSHUB ---
Run ID:       ce917c70cbbc4a70a6c4cc0be0ad12d0
mAP@0.5:      0.7159
mAP@0.5:0.95: 0.3567
